In [307]:
import glfw
from OpenGL.GL import *
import numpy as np
import math


Shaders - "attribute" para dados por vértice, "uniform" para dados globais

In [308]:
vertex_code = """
    attribute vec3 position;
    uniform mat4 mat_transformation;
    void main(){
        gl_Position = mat_transformation * vec4(position, 1.0);
    }
"""

fragment_code = """
    uniform vec4 color;
    void main(){
        gl_FragColor = color;
    }
"""

Inicializacao da Janela

In [309]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)
window = glfw.create_window(700, 700, "Dois Objetos", None, None)
glfw.make_context_current(window)

### Compilação e links dos shader

In [310]:
program  = glCreateProgram()
vertex   = glCreateShader(GL_VERTEX_SHADER)
fragment = glCreateShader(GL_FRAGMENT_SHADER)

glShaderSource(vertex, vertex_code)
glShaderSource(fragment, fragment_code)

glCompileShader(vertex)
if not glGetShaderiv(vertex, GL_COMPILE_STATUS):
    raise RuntimeError(glGetShaderInfoLog(vertex).decode())

glCompileShader(fragment)
if not glGetShaderiv(fragment, GL_COMPILE_STATUS):
    raise RuntimeError(glGetShaderInfoLog(fragment).decode())

glAttachShader(program, vertex)
glAttachShader(program, fragment)
glLinkProgram(program)
glUseProgram(program)

In [311]:
# Geração da semiesfera — copiado do notebook, sem alteração
# Isso substitui completamente o vertices_list manual que tínhamos
PI = 3.141592
r = 0.5
num_sectors = 32
num_stacks = 32

sector_step = (PI * 2) / num_sectors
stack_step  = PI / num_stacks

def F(u, v, r):
    x = r * math.sin(v) * math.cos(u)
    y = r * math.sin(v) * math.sin(u)
    z = r * math.cos(v)
    return (x, y, z)

vertices_list = []
for i in range(num_sectors):
    for j in range(num_stacks):
        u  = i * sector_step
        v  = j * stack_step
        un = PI * 2 if i + 1 == num_sectors else (i + 1) * sector_step
        vn = PI     if j + 1 == num_stacks  else (j + 1) * stack_step

        p0, p1, p2, p3 = F(u,v,r), F(u,vn,r), F(un,v,r), F(un,vn,r)

        vertices_list.append(p0)
        vertices_list.append(p2)
        vertices_list.append(p1)
        vertices_list.append(p3)
        vertices_list.append(p1)
        vertices_list.append(p2)

# Filtra apenas a metade inferior (z <= 0), como o notebook faz
total = len(vertices_list)
vertices = np.zeros(total, [("position", np.float32, 3)])
vertices["position"] = vertices_list
vertices = vertices[vertices["position"][:, 2] <= 0] # z <= 0 é a metade inferior da esfera
SEMIESFERA_INICIO = 0
SEMIESFERA_QTDE   = len(vertices)  # quantos vértices sobraram após o filtro

In [312]:
# ─────────────────────────────────────────────────────────────
# Envio dos dados para a GPU
# ─────────────────────────────────────────────────────────────
buffer_VBO = glGenBuffers(1)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_DYNAMIC_DRAW)

stride = vertices.strides[0]
offset = ctypes.c_void_p(0)

loc_position = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc_position)
glVertexAttribPointer(loc_position, 3, GL_FLOAT, False, stride, offset)

# Pegamos as localizações das uniforms uma vez só — mais eficiente
loc_color      = glGetUniformLocation(program, "color")
loc_transform  = glGetUniformLocation(program, "mat_transformation")

In [313]:
def mat_rotacao_x(angulo):
    c, s = math.cos(angulo), math.sin(angulo)
    return np.array([
        1.0, 0.0, 0.0, 0.0,  # X não muda — estamos girando em torno dele
        0.0,   c,  -s, 0.0,  # Y e Z são afetados
        0.0,   s,   c, 0.0,
        0.0, 0.0, 0.0, 1.0,
    ], np.float32)

def mat_rotacao_y(angulo):
    c, s = math.cos(angulo), math.sin(angulo)
    return np.array([
          c, 0.0,   s, 0.0,  # X e Z são afetados
        0.0, 1.0, 0.0, 0.0,  # Y não muda — estamos girando em torno dele
         -s, 0.0,   c, 0.0,
        0.0, 0.0, 0.0, 1.0,
    ], np.float32)


def mat_rotacao_z(angulo):
    c, s = math.cos(angulo), math.sin(angulo)
    return np.array([
        c,  -s, 0.0, 0.0,
        s,   c, 0.0, 0.0,
        0.0, 0.0, 1.0, 0.0,
        0.0, 0.0, 0.0, 1.0,
    ], np.float32)

def mat_translacao(tx, ty, tz):
    return np.array([
        1.0, 0.0, 0.0,  tx,
        0.0, 1.0, 0.0,  ty,
        0.0, 0.0, 1.0,  tz,
        0.0, 0.0, 0.0, 1.0,
    ], np.float32)

def mat_identidade():
    return np.eye(4, dtype=np.float32).reshape(1, 16)

In [314]:
angulo = 0.0
angulo_abertura = 0.0
import random

In [315]:

glEnable(GL_DEPTH_TEST)
glfw.show_window(window)

while not glfw.window_should_close(window):
    angulo += 0.01
    angulo_abertura += 0.05
    # Ângulos evoluem em velocidades diferentes — fica visualmente claro
    # que cada objeto tem sua própria transformação independente

    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    glClearColor(0.15, 0.15, 0.2, 1.0)

    # Semiesfera 1
    rot = mat_rotacao_x(math.radians(90 + angulo_abertura)).reshape(4, 4)
    rot_y = mat_rotacao_y(angulo).reshape(4, 4)
    rot = rot_y @ rot
    tra = mat_translacao(0.0, 0.0, 0.0-angulo_abertura/150).reshape(4, 4)
    mat = (rot@tra).reshape(1, 16)
    glUniformMatrix4fv(loc_transform, 1, GL_TRUE, mat)

    for triangle in range(0, len(vertices), 3):
        random.seed(triangle)   # seed fixo por triângulo — a cor não muda entre frames
        R = random.random()
        G = random.random()
        B = random.random()
        glUniform4f(loc_color, 1.0, B-0.2, G-0.2, 1.0)
        glDrawArrays(GL_TRIANGLES, triangle, 3)  # desenha 1 triângulo por vez

    #glUniform4f(loc_color, 1.0, 1.0, 1.0, 1.0)  # branco
    #glDrawArrays(GL_TRIANGLES, SEMIESFERA_INICIO, SEMIESFERA_QTDE)

    #semiesfera 2
    #rot = mat_rotacao_z(-angulo).reshape(4, 4)
    rot = mat_rotacao_x(math.radians(-90)).reshape(4, 4)
    rot_y = mat_rotacao_y(angulo).reshape(4, 4)
    rot = rot_y @ rot
    tra = mat_translacao(0.0, 0.0, 0.1).reshape(4, 4)
    mat = (rot @ tra).reshape(1, 16)
    glUniformMatrix4fv(loc_transform, 1, GL_TRUE, mat)

    #glUniform4f(loc_color, 1.0, 0.0, 0.0, 1.0)  # vermelho

    for triangle in range(0, len(vertices), 3):
        random.seed(triangle)   # seed fixo por triângulo — a cor não muda entre frames
        R = random.random()
        G = random.random()
        B = random.random()
        glUniform4f(loc_color, 1.0, 1.0, 1.0, 1.0)
        glDrawArrays(GL_TRIANGLES, triangle, 3)  # desenha 1 triângulo por vez

    glDrawArrays(GL_TRIANGLES, SEMIESFERA_INICIO, SEMIESFERA_QTDE)

    glfw.swap_buffers(window)
    glfw.poll_events()

glfw.terminate()